# RunnableParallel

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [4]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

In [5]:
chat = ChatOpenAI(
    model_name='llama3.2:3b',
    openai_api_key='ollama', 
    openai_api_base='http://localhost:11434/v1',
    temperature = 0, 
    max_tokens = 100,
    model_kwargs = {
        'seed':365
    }
)

/opt/anaconda3/envs/ai-ml/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3519: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [6]:
string_parser = StrOutputParser()

In [7]:
chain_books = chat_template_books | chat | string_parser
chain_projects = chat_template_projects | chat | string_parser

In [8]:
chain_parallel = RunnableParallel({
    'books' : chain_books,
    'projects' : chain_projects
})

In [9]:
chain_parallel.invoke({
    'programming language':'Python'
})

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests\n2. Creating a Chatbot with Natural Language Processing (NLP) using NLTK and Keras\n3. Developing a Game of Tic-Tac-Toe using Minimax Algorithm and GUI with Tkinter or PyQt'}

In [10]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [11]:
%%time
chain_books.invoke({'programming language':'Python'})

CPU times: user 6.05 ms, sys: 2.37 ms, total: 8.41 ms
Wall time: 990 ms


'1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz'

In [12]:
%%time
chain_projects.invoke({'programming language':'Python'})

CPU times: user 4.01 ms, sys: 1.75 ms, total: 5.77 ms
Wall time: 1.24 s


'1. Building a Web Scraper using BeautifulSoup and Requests\n2. Creating a Chatbot with Natural Language Processing (NLP) using NLTK and Keras\n3. Developing a Game of Tic-Tac-Toe using Minimax Algorithm and GUI with Tkinter or PyQt'

In [13]:
%%time
chain_parallel.invoke({'programming language':'Python'})

CPU times: user 10.6 ms, sys: 2.95 ms, total: 13.5 ms
Wall time: 2.16 s


{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests\n2. Creating a Chatbot with Natural Language Processing (NLP) using NLTK and Keras\n3. Developing a Game of Tic-Tac-Toe using Minimax Algorithm and GUI with Tkinter or PyQt'}